# 迹线 Transformer 涡提取 —— Kaggle 训练（票 07）

在本 Notebook 完成 `pipedcylinder2d.nc` 上 200 epoch 全量训练（Kaggle T4×2、12h 会话硬上限 → ≤8h 分块 + 每 epoch checkpoint + 跨会话断点续训）。

**运行前准备（详见 `kaggle/README.md`）**：
1. Notebook Settings：Accelerator = GPU T4 x2、Internet = ON；
2. Add Input：Dataset A（`kaggle/prepare_dataset_a.py` 打包产物，含 `dataset/meta.json`）；
3.（可选，模式 A）Add-ons → Secrets 添加 `KAGGLE_USERNAME`/`KAGGLE_KEY`，并在下方 `CKPT_DATASET_SLUG` 填 checkpoint 数据集 slug（跨会话自动发布/下载）；
4.（可选，模式 B）把上一会话的 checkpoint 打包文件上传为 Kaggle Dataset 并 Add Input，或从 `/kaggle/working` 下载后重新上传。

**cell 直接顺序执行（Run All）**。会话结束（12h）后重启会话再 Run All 即从 latest checkpoint 续训。

In [ ]:
import os
# Kaggle T4 16GB：CUDA 分配碎片缓解（OOM 提示建议；子进程继承）
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("PYTHONUNBUFFERED", "1")  # 子进程 print 实时刷（管道下默认块缓冲会滞后/吞掉日志）
import subprocess
import subprocess
subprocess.run(["pip", "install", "-q", "h5py", "PyYAML", "matplotlib", "tqdm", "kaggle"], check=True)
import sys, torch, numpy, h5py, yaml
print("python", sys.version.split()[0], "| torch", torch.__version__,
      "| cuda:", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())

In [ ]:
import os, pathlib, shutil
REPO_URL = "https://github.com/ziyixu317-wq/2d-vortex-extraction-260825.git"
REPO_DIR = "/kaggle/working/repo"
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)      # 每次克隆最新 main（/kaggle/working 跨会话残留旧仓库会导致漏更——
                                 #   「仓库已就绪」快路径曾让旧代码复用，改强制重克隆）
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)   # Script/Notebook 双保险（import dataset 等）
head = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                      capture_output=True, text=True).stdout.strip()
print("repo HEAD:", head, "（需 >= b3c381e；更旧说明克隆失败/推送未达）")
print("repo 根文件:", sorted(os.listdir(REPO_DIR))[:12])
print("cwd:", os.getcwd())


In [ ]:
# 挂载布局自适应（票 07 延伸：单数据集 / 多数据集两种 Dataset A 布局自动探测）
#   单： <挂载根>/<nc> + <挂载根>/dataset/meta.json
#   多： <挂载根>/data/<nc> + <挂载根>/datasets/<名>/dataset/meta.json
import glob, shutil, zipfile

try:
    from kaggle.mount_probe import probe_layout   # 多级嵌套挂载（datasets/<owner>/<slug>/）自动命中
except ModuleNotFoundError:
    print("[env] 导入 kaggle.mount_probe 失败——仓库未克隆到最新，或 kaggle 名被 site-packages CLI"
          "包遮蔽。请先 Run cell 2（强制重克隆 + 打印 repo HEAD），确认 HEAD >= b3c381e 后再 Run All。")
    raise
print("[env] /kaggle/input 挂载:", [p.name for p in pathlib.Path("/kaggle/input").glob("*")])
single, multi = probe_layout()
if single is None and multi is None:
    zips = list(pathlib.Path("/kaggle/input").rglob("*.zip"))
    if zips:
        print(f"[env] 未直接命中；检测到 zip（未解压?）→ 解压后重找: {[z.name for z in zips]}")
        ex = pathlib.Path("/kaggle/working/dataset_a_extract")
        ex.mkdir(parents=True, exist_ok=True)
        for z in zips:
            with zipfile.ZipFile(z) as zf:
                zf.extractall(ex)
        single, multi = probe_layout(str(ex))
if single is None and multi is None:
    print("[env] 未找到 Dataset A。输入树（/kaggle/input 全览）：")
    for d in pathlib.Path("/kaggle/input").rglob("*"):
        print("  ", d)
    raise AssertionError("未找到 Dataset A：单数据集布局（dataset/meta.json）或"
                         "多数据集布局（datasets/<名>/dataset/meta.json）——检查 Add Input 挂载")

# ---- 布局 → 训练配置（单/多自动选择；多数据集 = pathline_transformer_multi.yaml，票 07 延伸）
if multi is not None:
    CONFIG = "config/pathline_transformer_multi.yaml"    # 7 roots、frac 60/40
    print("[env] 多数据集布局 → config:", CONFIG)
    # 每个 datasets/<名>/ → outputs/datasets/<名>/（Kaggle slug 与本地目录名一致化）
    for src in sorted((multi / "datasets").iterdir()):
        if not (src / "dataset").exists():
            continue
        dst = pathlib.Path("outputs/datasets") / src.name
        if dst.exists():
            shutil.rmtree(dst)
        dst.parent.mkdir(parents=True, exist_ok=True)
        try:
            os.symlink(src, dst)
            print("  链接", src, "→", dst)
        except OSError:
            shutil.copytree(src, dst)
            print("  复制", src, "→", dst)
    if os.path.exists("outputs/dataset"):
        shutil.rmtree("outputs/dataset")      # 防旧单数据集残留混入
else:
    CONFIG = "config/pathline_transformer_cylinder.yaml"
    print("[env] 单数据集布局 → config:", CONFIG)
    dst = pathlib.Path("outputs/dataset")
    if dst.exists():
        shutil.rmtree(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    src_ds = single / "dataset"
    try:
        os.symlink(src_ds, dst)
        print("数据已链接 →", src_ds)
    except OSError:
        shutil.copytree(src_ds, dst)
        print("数据已复制 →", dst)

TRAIN_CONFIG = CONFIG              # 训练配置（块 5 校准可能生成优化的覆盖配置）
cfg0 = yaml.safe_load(open(CONFIG, encoding="utf-8"))
TOTAL_EPOCHS = int(cfg0["train"]["epochs"])     # 200（HANDOFF §6；以 config 为单一来源）
RUN_NAME = str(cfg0["train"].get("run_name", "run"))   # 单/多配置的 run_name（cell 6/8/9 复用）
CHUNK_BUDGET_H = 7.5              # 每块会话预算（12h 上限内留自检/打包余量）
CKPT_DATASET_SLUG = ""           # 模式 A：checkpoint 数据集 slug（如 "yourname/vortex-train-ckpt"）；留空 = 手动模式

def find_input(marker, root="/kaggle/input"):
    """在挂载树中定位含 marker 的挂载根目录（返回该根 Path；含 marker 时不得 None）。"""
    for d in sorted(pathlib.Path(root).glob("*")):
        if (d / marker).exists():
            return d
    for hit in pathlib.Path(root).rglob(marker):
        return hit.parents[len(marker.split("/")) - 1]   # 任意深度 → 回到挂载根
    return None

# ---- checkpoint 数据集（跨会话续训；模式 A 自动下载 / 模式 B 手动挂载 input）
if CKPT_DATASET_SLUG:
    subprocess.run(["kaggle", "datasets", "download", CKPT_DATASET_SLUG,
                    "-p", "/tmp/ckpt", "--unzip"], check=True)
    ckpt_src = pathlib.Path("/tmp/ckpt")
else:
    ckpt_src = find_input(f"{RUN_NAME}_ckpt_latest.pth")
if ckpt_src is not None:
    os.makedirs("outputs/train", exist_ok=True)
    for p in pathlib.Path(ckpt_src).glob("*"):
        if p.is_file():
            shutil.copy2(p, pathlib.Path("outputs/train") / p.name)
    print("checkpoint 已还原 → outputs/train/（注意：若本次是从头重训/新 τ 标签，请勿挂载旧 checkpoint 数据集，"
          "否则 --resume auto 会从旧权重续训）")
else:
    print("首次会话：无 checkpoint，从零开始（--resume auto 自动处理）")


In [ ]:
# 验收 1：Notebook 环境 import vendor + 数据加载通过（失败会 raise，不静默）
# 自检数据根随布局自适应：单数据集 = outputs/dataset；多数据集 = cell 3 链接的任一数据集
#（多布局下无 outputs/dataset——票 07 延伸；取自检目录 outputs/datasets/pipedcylinder2d/dataset，
#  其余数据集同构，自检语义 = 「vendor 导入 + 模型前向 + on-the-fly 样本有限」环境级验证）
if os.path.isdir("outputs/dataset"):
    CHECK_ROOT = "outputs/dataset"
else:
    multi_roots = sorted(pathlib.Path("outputs/datasets").glob("*/dataset"))
    if not multi_roots:
        raise AssertionError("多数据集模式未找到 outputs/datasets/<名>/dataset——请先运行 cell 3（挂载链接）")
    CHECK_ROOT = str(pathlib.Path("outputs/datasets/pipedcylinder2d/dataset")
                      if (pathlib.Path("outputs/datasets/pipedcylinder2d/dataset").exists())
                      else multi_roots[0])
print("[自检] data-root =", CHECK_ROOT)
subprocess.run([sys.executable, "kaggle/self_check.py",
                "--data-root", CHECK_ROOT,
                "--config", CONFIG, "--device", "cuda", "--n-samples", "4"], check=True, cwd=REPO_DIR)


In [ ]:
# 验收 2：1 epoch 实测步速（首会话真测；后续会话复用 bench_info.json，不重跑校准）
import time, json, os
BENCH_INFO = "outputs/bench_info.json"
BENCH_RESTORED = "outputs/train/bench_info.json"   # 上个会话块尾打包 → cell 3 还原（跨会话免重复校准）
from kaggle.chunking import pick_bench_source
bench_src, bench = pick_bench_source(BENCH_INFO, BENCH_RESTORED)
if bench_src is not None:
    print(f"复用步速基准（{bench_src}）: 1 epoch = {bench['seconds_per_epoch']:.1f} s（{bench['timestamp']} 实测，省 ~18min 校准）")
else:
    BENCH_YAML = "outputs/bench_config.yaml"
    cfg = yaml.safe_load(open(CONFIG, encoding="utf-8"))
    cfg["train"]["epochs"] = 1
    cfg["train"]["ckpt_dir"] = "outputs/bench"
    cfg["train"]["run_name"] = "bench"
    cfg["data"]["val_split"] = "none"   # 校准不做 val：val 评估 20000 样本 ≈17.5min 且无进度条（曾致"卡死"误判）
    yaml.safe_dump(cfg, open(BENCH_YAML, "w", encoding="utf-8"), allow_unicode=True)
    t0 = time.time()
    subprocess.run([sys.executable, "train_kaggle.py", "--config", BENCH_YAML,
                    "--resume", "none"], check=True, cwd=REPO_DIR)
    seconds_per_epoch = time.time() - t0
    bench = {"seconds_per_epoch": seconds_per_epoch,
             "samples_per_epoch": cfg["data"]["samples_per_epoch"],
             "steps_per_epoch": int(cfg["data"]["samples_per_epoch"] / cfg["data"]["batch_size"]),
             "timestamp": time.strftime("%Y-%m-%d %H:%M")}
    json.dump(bench, open(BENCH_INFO, "w", encoding="utf-8"), indent=2)
    print(f"1 epoch 实测 = {seconds_per_epoch:.1f} s（{bench['steps_per_epoch']} 步，{bench['samples_per_epoch']} 样本）")

# ---- 校准：总时长预算检查（HANDOFF §7 风险预案：超预算启用 DataParallel/AMP/降样本数）
sps = float(bench["seconds_per_epoch"])
total_h = sps * TOTAL_EPOCHS / 3600
print(f"200 epoch 预计总时长 ≈ {total_h:.1f} h（目标 ≤ 4 个 12h 会话）")
if total_h > 4 * 12:
    opt = yaml.safe_load(open(CONFIG, encoding="utf-8"))
    opt["train"]["data_parallel"] = torch.cuda.device_count() > 1
    n_samples = max(20000, int(opt["data"]["samples_per_epoch"] * 0.5))
    opt["data"]["samples_per_epoch"] = n_samples     # HANDOFF §6 下限 20000
    yaml.safe_dump(opt, open("outputs/train_opt.yaml", "w", encoding="utf-8"),
                    allow_unicode=True)
    TRAIN_CONFIG = "outputs/train_opt.yaml"
    print(f"超预算：已生成 {TRAIN_CONFIG}（样本数→{n_samples}；AMP 不启用——上游模型 propagate_features 与 Half 不兼容，见 README）")
    print("回填 HANDOFF §6：以实测校准 samples_per_epoch（下限 20000）；步速/时长见本 cell")
else:
    TRAIN_CONFIG = CONFIG
    print("预算内：直接用生产配置")

from kaggle.chunking import plan_chunks
plan = plan_chunks(TOTAL_EPOCHS, sps, CHUNK_BUDGET_H * 3600)
print(f"{CHUNK_BUDGET_H}h 预算 → 每块最多 {max(plan)} epoch；分块计划 {plan}（约 {len(plan)} 个会话）")
print(f"本块训练配置: {TRAIN_CONFIG}")

In [ ]:
# 验收 3：分块训练（每会话一块；--resume auto 从 latest 无损伤续训）
import torch, pathlib as _pl
from kaggle.chunking import plan_chunks
bench = json.load(open("outputs/bench_info.json", encoding="utf-8"))
sps = float(bench["seconds_per_epoch"])
latest = _pl.Path(f"outputs/train/{RUN_NAME}_ckpt_latest.pth")
def progress():
    if not latest.exists():
        return 0
    return int(torch.load(latest, map_location="cpu")["epoch"]) + 1

p = progress()
assert p < TOTAL_EPOCHS, f"训练已完成（已到 {p} epoch）——直接看收尾 cell"
plan = plan_chunks(TOTAL_EPOCHS - p, sps, CHUNK_BUDGET_H * 3600)
chunk = plan[0]
target = p + chunk
cmd = [sys.executable, "train_kaggle.py", "--config", TRAIN_CONFIG,
       "--resume", "auto", "--epochs", str(target)]
if target >= TOTAL_EPOCHS:
    cmd.append("--report-f1")      # 最后一块：训练完成后记录 val F1（验收 4）
print(f"[分块] 从 epoch {p} 续到 {target}（本块 {chunk}；完整计划 {plan}）")
subprocess.run(cmd, check=True, cwd=REPO_DIR)
print(f"[分块] 本块完成：此会话结束（进度 {progress()}/{TOTAL_EPOCHS}）。"
      "重启会话再次 Run All 即续训）")


In [ ]:
import zipfile, shutil
# 校准结果随 checkpoint 一起打包 → 下个会话 cell 3 还原后复用（省 ~18min 重复校准）
if os.path.exists("outputs/bench_info.json"):
    shutil.copy2("outputs/bench_info.json", pathlib.Path("outputs/train/bench_info.json"))
    print("bench_info.json 已随 checkpoint 打包（下个会话复用）")
# 块尾：checkpoint 打包（跨会话续训载体）
srcdir = pathlib.Path("outputs/train")
zpath = pathlib.Path("/kaggle/working") / "ckpt_snapshot.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(srcdir.glob("*")):
        if p.is_file():
            zf.write(p, p.name)
print("checkpoint 快照已打包:", zpath, f"({zpath.stat().st_size/1e6:.1f} MB)")
if CKPT_DATASET_SLUG:
    # 模式 A：发布为 checkpoint 数据集新版本（下次会话自动下载续训）
    subprocess.run(["kaggle", "datasets", "version", CKPT_DATASET_SLUG,
                    "-p", str(srcdir), "--dir-mode", "zip",
                    "-m", f"train chunk to epoch {progress()-1}"], check=True)
    print("已发布 checkpoint 数据集新版本 →", CKPT_DATASET_SLUG)
else:
    # 模式 B：手动 —— 下载 zpath，上传为 Kaggle Dataset（挂载为 input 供下次会话）
    print("手动模式：下载 ckpt_snapshot.zip → Kaggle 新建 Dataset 上传 → 下次会话 Add Input")

In [ ]:
# 验收 4：训练完成收尾（val F1 记录 + 最终 checkpoint 归档）
f1_files = sorted(pathlib.Path("outputs/train").glob("*_f1.json"))
if f1_files:
    print("F1/IoU 记录（自然分布，单配置 val / 多配置 test）:")
    print(json.dumps(json.loads(f1_files[0].read_text(encoding="utf-8")), indent=2))
else:
    print("尚未训练完成（无 *_f1.json）——继续跑分块 cell 直至 200 epoch")

import hashlib
final_zip = pathlib.Path("/kaggle/working") / "final_ckpt.zip"
with zipfile.ZipFile(final_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(srcdir.glob("*")):
        if p.is_file():
            zf.write(p, p.name)
h = hashlib.sha256(final_zip.read_bytes()).hexdigest()
print(f"最终 checkpoint 归档: {final_zip} ({final_zip.stat().st_size/1e6:.1f} MB)\nsha256 = {h}")
print("步骤：下载 final_ckpt.zip → 本地 outputs/archive/ → 回填票文件（val F1/步速/checkpoint 位置）")


In [ ]:
# （可选）中途预览：单帧 模型概率 vs IVD vs 弱标签 三联图（正式评估属票 08）
# 任意会话随时可跑：加载 latest checkpoint → 帧 1300（test 片）滑窗 → TTA3 → 投影
subprocess.run([sys.executable, "kaggle/preview_eval.py",
                "--config", CONFIG,
                "--ckpt", f"outputs/train/{RUN_NAME}_ckpt_latest.pth",
                "--frame", "1300", "--out", "outputs/preview/prob_vs_ivd_t1300.png",
                "--device", "cuda", "--tta", "3"], check=True, cwd=REPO_DIR)
